# Deformation summary

Collects `images/total_deformation.png` from every run folder in `outputs/` and merges the runs of the same case (same configuration, different pedal box positions) into one image per case, saved to `deformation summary/`.

Each tile is the Mechanical image as exported, so **each tile has its own colour scale** (read the legend in the tile). The tile title gives the pedal box position and, when available in `results.csv`, the maximum total deformation.

In [ ]:
import json
import math
import pathlib
from collections import defaultdict

import matplotlib

matplotlib.use("Agg")  # write files only; the summary images are shown at the end
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = pathlib.Path.cwd()
OUTPUTS = ROOT / "outputs"
SUMMARY = ROOT / "deformation summary"
IMAGE = "total_deformation.png"
IN = 25.4  # mm per inch; params.json stores lengths in mm
COLUMNS = 3  # tiles per row

In [ ]:
# Group run folders by case id. params.json holds each run's full configuration.
runs = defaultdict(list)
for params_file in sorted(OUTPUTS.glob("case_*_off*/params.json")):
    image = params_file.parent / "images" / IMAGE
    if not image.exists():
        print(f"skipped (no {IMAGE}): {params_file.parent.name}")
        continue
    p = json.loads(params_file.read_text())
    runs[p["case_id"]].append((p["pedal_offset_mm"] / IN, image, p))

# Optional: max deformation per run from results.csv (any units it was written in).
results = pd.read_csv(OUTPUTS / "results.csv") if (OUTPUTS / "results.csv").exists() else pd.DataFrame()
def_column = next((c for c in results.columns if c.startswith("max_total_def_")), None)


def max_deformation(case_id, offset_in):
    if def_column is None:
        return ""
    row = results[(results["case_id"].astype(int) == case_id) & ((results["pedal_offset_in"] - offset_in).abs() < 1e-3)]
    unit = def_column.removeprefix("max_total_def_")
    return f"\nmax total deformation {row[def_column].iloc[0]:.4f} {unit}" if len(row) else ""


def config_label(case_id, p):
    """Two-line description of the configuration shared by all runs of a case."""
    line1 = (
        f"Case {case_id}:  hardpoint r = {p['hardpoint_rad'] / IN:.3f} in, {p['hardpoint_material']}   |   "
        f"core {p['core_material']}, {p['core_thickness'] / IN:.3f} in   |   edges {p['edge_bc']}   |   mesh {p['mesh_quality']}"
    )
    line2 = f"Layup {p['layup']}"
    return f"{line1}\n{line2}"


for case_id, items in sorted(runs.items()):
    print(f"case {case_id}: positions {[round(o, 2) for o, _, _ in sorted(items, key=lambda r: r[0])]} in")

In [ ]:
def trimmed(image_path, pad=10):
    """The image with its white border cut off, so the panel fills more of the tile."""
    img = mpimg.imread(image_path)
    content = (img[..., :3] < 0.98).any(axis=2)
    rows_, cols_ = content.any(axis=1).nonzero()[0], content.any(axis=0).nonzero()[0]
    top, bottom = max(rows_[0] - pad, 0), min(rows_[-1] + pad, img.shape[0])
    left, right = max(cols_[0] - pad, 0), min(cols_[-1] + pad, img.shape[1])
    return img[top:bottom, left:right]


SUMMARY.mkdir(exist_ok=True)
written = []
for case_id, items in sorted(runs.items()):
    items.sort(key=lambda r: r[0])
    n = len(items)
    cols = min(COLUMNS, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6.4 * cols, 4.4 * rows + 1.0), squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, (offset_in, image, p) in zip(axes.flat, items):
        ax.imshow(trimmed(image))
        ax.set_title(f"Pedal box offset {offset_in:.2f} in{max_deformation(case_id, offset_in)}", fontsize=12)
    fig.suptitle(config_label(case_id, items[0][2]), fontsize=14, fontweight="bold")
    fig.supxlabel("Total deformation. Each tile has its own colour scale; offsets are measured forward from the rearmost position.",
                  fontsize=10, style="italic")
    path = SUMMARY / f"case_{case_id:03d}_total_deformation.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    written.append(path)
    print(f"wrote {path.relative_to(ROOT)} ({n} positions)")

In [ ]:
for path in written:
    display(Image(filename=str(path)))